# Timing versus Intensity of Breeding-Site Clean-Up Campaigns for Dengue Control in Colombo

*Summary (to be completed when the results are in):* a host–vector dengue model (SEIR humans, SEI *Aedes* mosquitoes), solved with hand-written Euler and RK4 integrators, calibrated to weekly reported cases in Colombo District with a hand-written Metropolis MCMC sampler, and used to compare clean-up campaigns of different intensity and start week.

In [ ]:
import os, time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import scipy
from scipy import stats, optimize
from scipy.special import gammaln

SEED = 42                                  # global seed, used by every random step in this notebook
rng = np.random.default_rng(SEED)
os.makedirs("figures", exist_ok=True)
print("numpy", np.__version__, "| scipy", scipy.__version__, "| pandas", pd.__version__, "| matplotlib", matplotlib.__version__)

## 1. Problem and question

Dengue is endemic in Sri Lanka, and Colombo District is consistently among the most affected districts. Cases rise sharply after the south-west monsoon, as they did again in 2026. The main community response is breeding-site clean-up, which reduces the number of new adult *Aedes* mosquitoes. Health authorities must decide when to start a campaign and how intensive to make it.

**Question:** how early and how large must a breeding-site clean-up campaign be to halve the peak weekly dengue case count in Colombo District?

The answer is reported with posterior uncertainty bands, not single runs.

## 2. Model, assumptions and equations

**State variables.** Humans are tracked as fractions of the effective susceptible population $N$: susceptible $s$, exposed $e$, infectious $i$, recovered $r$, and cumulative incidence $c$. Mosquitoes are tracked per human: susceptible $V_s$, exposed $V_e$, infectious $V_i$.

$$
\begin{aligned}
\frac{ds}{dt} &= -\beta V_i s, &
\frac{dV_s}{dt} &= \lambda(t) - \beta i V_s - \mu_v V_s,\\
\frac{de}{dt} &= \beta V_i s - \sigma_h e, &
\frac{dV_e}{dt} &= \beta i V_s - (\sigma_v + \mu_v) V_e,\\
\frac{di}{dt} &= \sigma_h e - \gamma i, &
\frac{dV_i}{dt} &= \sigma_v V_e - \mu_v V_i,\\
\frac{dr}{dt} &= \gamma i, &
\frac{dc}{dt} &= \sigma_h e .
\end{aligned}
$$

**Clean-up campaign.** A campaign that starts on day $t_c$ cuts mosquito recruitment by a fraction $p$:

$$\lambda(t) = m_0 \mu_v \left(1 - p\,\mathbf{1}[t \ge t_c]\right).$$

**Basic reproduction number** (next-generation matrix, host → vector → host):

$$R_0^2 = \frac{\beta^2 m_0 \sigma_v}{\gamma\, \mu_v (\sigma_v + \mu_v)}, \qquad R_0(p)^2 = R_0^2 (1 - p), \qquad p_c = 1 - \frac{1}{R_0^2}.$$

A campaign applied from the start stops the outbreak exactly when $p > p_c$.

**Observation model** (used from section 5): weekly reported cases $y_w \sim \text{NegBin}\big(\mu_w = \rho N \Delta c_w,\ k\big)$.

| Symbol | Meaning | Units | Value | Source |
|---|---|---|---|---|
| $\sigma_h$ | 1 / human intrinsic incubation period | 1/day | 1/5.5 | [literature, to cite] |
| $\gamma$ | 1 / human infectious period | 1/day | 1/5 | [literature, to cite] |
| $\sigma_v$ | 1 / extrinsic incubation period in the mosquito | 1/day | 1/10 | [literature, to cite] |
| $\mu_v$ | adult *Aedes* death rate (1 / lifespan) | 1/day | 1/14 | [literature, to cite] |
| $m_0$ | baseline adult mosquitoes per human | – | 2 | assumed; varied in sensitivity |
| $N$ | effective susceptible population = susceptible fraction × district population | people | 0.30 × 2,375,415 | Census 2024; fraction assumed |
| $\beta$ | transmission rate (same in both directions) | 1/day | fitted | – |
| $\rho$ | reporting fraction | – | fitted | – |
| $I_0$ | infectious people at the start of the fitting window (week 20) | people | fitted | – |
| $k$ | negative-binomial dispersion | – | 20 | assumed |

**Assumptions**
1. Closed population over one season; births, deaths and migration of humans are ignored.
2. One circulating serotype with lasting immunity within the season.
3. Homogeneous mixing across the district.
4. The same transmission rate $\beta$ applies human → mosquito and mosquito → human.
5. A campaign cuts the recruitment of new adult mosquitoes; adults already alive die off at their natural rate.
6. The susceptible fraction is fixed (it trades off exactly with $\rho$ within one wave) and is varied in the sensitivity analysis.
7. The reporting fraction $\rho$ is constant over the season.

**Concepts used:** population and epidemic models (SEIR / SEI); numerical ODE solvers (Euler, RK4, error orders); MCMC and the Metropolis algorithm; Monte Carlo simulation and input modelling.

## 3. Implementation

### 3.1 Model and solvers

All solvers and the model are written from scratch with NumPy. The time step is $\Delta t = 0.5$ day.

In [ ]:
N_POP = 2_375_415        # Colombo District population, Census of Population and Housing 2024 (final report)
SUSC_FRAC = 0.30         # ASSUMED share of the population susceptible to the circulating serotype (varied in sensitivity)
N = SUSC_FRAC * N_POP    # effective susceptible population [people]

FIXED = dict(
    sigma_h = 1/5.5,     # 1 / human intrinsic incubation period   [1/day]
    gamma   = 1/5.0,     # 1 / human infectious period             [1/day]
    sigma_v = 1/10.0,    # 1 / extrinsic incubation in the mosquito [1/day]
    mu_v    = 1/14.0,    # 1 / adult Aedes lifespan                [1/day]
    m0      = 2.0,       # baseline adult mosquitoes per human     [-]
)
DT = 0.5                 # RK4 time step [day]
# state vector y = [s, e, i, r, c, Vs, Ve, Vi]
#   s, e, i, r : fractions of N (sum to 1);  c : cumulative incidence (fraction of N)
#   Vs, Ve, Vi : mosquitoes per human

In [ ]:
def rhs(t, y, beta, lam, P=FIXED):
    """Host-vector SEIR-SEI derivatives. lam = mosquito recruitment rate [mosquitoes per human per day]."""
    s, e, i, r, c, Vs, Ve, Vi = y
    sh, g, sv, mv = P["sigma_h"], P["gamma"], P["sigma_v"], P["mu_v"]
    inf_h = beta * Vi * s            # new human infections   [1/day]
    inf_v = beta * i * Vs            # new mosquito infections [1/day]
    return np.array([
        -inf_h,
        inf_h - sh * e,
        sh * e - g * i,
        g * i,
        sh * e,                      # cumulative incidence -> weekly cases
        lam - inf_v - mv * Vs,
        inf_v - (sv + mv) * Ve,
        sv * Ve - mv * Vi,
    ])

Both solvers are fixed-step. Explicit Euler has global error $O(\Delta t)$; classical RK4 has global error $O(\Delta t^4)$ for smooth right-hand sides.

In [ ]:
def euler(f, y0, t0, t1, dt):
    """Explicit Euler, fixed step. Returns times and states (rows)."""
    n = int(round((t1 - t0) / dt))
    ts = t0 + dt * np.arange(n + 1)
    ys = np.empty((n + 1, len(y0))); ys[0] = y0
    for k in range(n):
        ys[k + 1] = ys[k] + dt * f(ts[k], ys[k])
    return ts, ys

def rk4(f, y0, t0, t1, dt):
    """Classical fourth-order Runge-Kutta, fixed step. Returns times and states (rows)."""
    n = int(round((t1 - t0) / dt))
    ts = t0 + dt * np.arange(n + 1)
    ys = np.empty((n + 1, len(y0))); ys[0] = y0
    for k in range(n):
        t, y = ts[k], ys[k]
        k1 = f(t, y)
        k2 = f(t + dt/2, y + dt/2 * k1)
        k3 = f(t + dt/2, y + dt/2 * k2)
        k4 = f(t + dt, y + dt * k3)
        ys[k + 1] = y + dt/6 * (k1 + 2*k2 + 2*k3 + k4)
    return ts, ys

The campaign makes $\lambda(t)$ jump at $t_c$. If a solver step straddled the jump, RK4 would lose its fourth-order accuracy. `simulate` therefore integrates in two segments that meet exactly at $t_c$, which must lie on the time grid.

In [ ]:
def initial_state(I0, m=None, P=FIXED):
    """I0 infectious people at t = 0; mosquitoes start uninfected at density m (default m0)."""
    m = P["m0"] if m is None else m
    i0 = I0 / N
    return np.array([1 - i0, 0.0, i0, 0.0, 0.0, m, 0.0, 0.0])

def simulate(beta, I0, n_weeks, p=0.0, t_c=np.inf, dt=DT, solver=rk4, y0=None, P=FIXED):
    """Run the model for n_weeks. A clean-up campaign cuts mosquito recruitment by fraction p from day t_c.
    Integrated in two segments split exactly at t_c, so no solver step straddles the jump in recruitment."""
    T = 7 * n_weeks
    y0 = initial_state(I0, P=P) if y0 is None else y0
    lam0 = P["m0"] * P["mu_v"]
    t_c = min(max(t_c, 0.0), T)
    assert abs(t_c / dt - round(t_c / dt)) < 1e-9, "t_c must lie on the time grid"
    segs = [(0.0, t_c, lam0), (t_c, T, lam0 * (1 - p))]
    ts_all, ys_all = [np.array([0.0])], [y0[None, :]]
    y = y0
    for a, b, lam in segs:
        if b - a < 1e-12:
            continue
        ts, ys = solver(lambda t, yy: rhs(t, yy, beta, lam, P), y, a, b, dt)
        ts_all.append(ts[1:]); ys_all.append(ys[1:]); y = ys[-1]
    return np.concatenate(ts_all), np.vstack(ys_all)

def weekly_cases(ts, ys, rho):
    """Expected reported cases per week = rho * N * (increase in cumulative incidence over the week)."""
    week_idx = [np.argmin(abs(ts - 7 * w)) for w in range(int(round(ts[-1] / 7)) + 1)]
    c_week = ys[week_idx, 4]
    return rho * N * np.diff(c_week)

def R0(beta, p=0.0, P=FIXED):
    """Basic reproduction number (host -> vector -> host, square root of the two-generation product)."""
    m = P["m0"] * (1 - p)
    return np.sqrt(beta**2 * m * P["sigma_v"] / (P["gamma"] * P["mu_v"] * (P["sigma_v"] + P["mu_v"])))

def p_crit(beta, P=FIXED):
    """Smallest recruitment cut that stops an outbreak: R0(p)^2 = R0^2 (1 - p) = 1."""
    return 1 - 1 / R0(beta, P=P)**2

BETA_TEST, I0_TEST = 0.35, 50
ts, ys = simulate(BETA_TEST, I0_TEST, n_weeks=25)
wk = weekly_cases(ts, ys, rho=0.1)
print(f"R0 = {R0(BETA_TEST):.3f}, p_c = {p_crit(BETA_TEST):.3f}")
print(f"peak weekly cases (rho = 0.1) = {wk.max():.0f} in week {wk.argmax() + 1}")
t0 = time.perf_counter()
for _ in range(20): simulate(BETA_TEST, I0_TEST, n_weeks=25)
print(f"one 25-week RK4 solve: {(time.perf_counter() - t0) / 20 * 1000:.1f} ms")

### 3.2 Data

Weekly notified dengue cases for Colombo District, 2026, from the Epidemiology Unit's *Weekly Epidemiological Report* (Table 1), as extracted in the `denguedatahub` package; see `data/README.md`. Five weeks with extraction errors are marked missing and are skipped by the likelihood, not filled in.

**Fitting window: weeks 20–36 (11 May – 6 Sep 2026).** Weeks 16–19 are a flat pre-monsoon period (about 250–330 cases per week). A model with one constant transmission rate cannot stay flat and then rise steeply, so including them biases the fit: with a start at week 16, only 78% of observed weeks fell inside the 95% predictive band at the best-fit parameters, against 93% for week 19 and 100% for weeks 20 and 21. The window therefore covers the monsoon wave itself.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/KavinduG13/dengue-cleanup-simulator/main/data/colombo_weekly_dengue_2026.csv"
df = pd.read_csv(DATA_URL, parse_dates=["week_start", "week_end"])
FIT_START, FIT_END = 20, 36                        # fitting window (epi weeks): the monsoon wave, see the Markdown cell above
fit = df[(df.epi_week >= FIT_START) & (df.epi_week <= FIT_END)].reset_index(drop=True)
N_WEEKS = len(fit)
y_obs = fit.cases.to_numpy(dtype=float)            # NaN where the week is missing
obs_mask = ~np.isnan(y_obs)
print(f"Fitting window: weeks {FIT_START}-{FIT_END} ({N_WEEKS} weeks, {obs_mask.sum()} observed, "
      f"{(~obs_mask).sum()} missing: {fit.epi_week[~obs_mask].tolist()})")

fig, ax = plt.subplots(figsize=(8, 3.8))
ok = df[df.status == "ok"]
ax.plot(ok.week_start, ok.cases, "o-", ms=4, label="reported cases")
for t in df.week_start[df.status == "missing"]:
    ax.axvline(t, color="0.8", lw=5, alpha=0.6)
ax.axvspan(pd.Timestamp("2026-06-15"), pd.Timestamp("2026-06-21"), color="orange", alpha=0.3, label="control week 15-21 Jun")
ax.axvspan(fit.week_start.iloc[0], fit.week_end.iloc[-1], color="tab:blue", alpha=0.06, label="fitting window")
ax.set_ylabel("cases per week"); ax.set_title("Colombo District, 2026 (grey = missing week)"); ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### 3.3 Starting state

At the start of the window the epidemic is already under way. Starting with no exposed people and no infected mosquitoes would create an artificial lag of about two weeks. Instead, the infected compartments $(e, i, V_e, V_i)$ start on the dominant eigenvector of the linearised system $\dot x = (F - V)x$, scaled so that $I_0$ people are infectious: this is the mix of compartments that a growing epidemic settles into.

In [ ]:
def infected_jacobian(beta, P=FIXED):
    """Linearised dynamics of (e, i, Ve, Vi) near the disease-free state: dx/dt = (F - V) x."""
    sh, g, sv, mv, m = P["sigma_h"], P["gamma"], P["sigma_v"], P["mu_v"], P["m0"]
    return np.array([[-sh,  0.0,     0.0,          beta],
                     [ sh, -g,       0.0,          0.0 ],
                     [0.0,  beta*m, -(sv + mv),    0.0 ],
                     [0.0,  0.0,     sv,          -mv  ]])

def initial_state_growth(beta, I0, P=FIXED):
    """Start on the dominant eigenvector, so infected humans and mosquitoes are already in the
    proportions of a growing (or shrinking) epidemic, with I0 infectious people. This removes the
    artificial lag of starting with no exposed people and no infected mosquitoes."""
    w, v = np.linalg.eig(infected_jacobian(beta, P))
    vec = np.abs(np.real(v[:, np.argmax(np.real(w))]))
    e0, i0, Ve0, Vi0 = vec / vec[1] * (I0 / N)
    return np.array([1 - e0 - i0, e0, i0, 0.0, 0.0, P["m0"] - Ve0 - Vi0, Ve0, Vi0])

def expected_cases(theta, n_weeks=N_WEEKS, p=0.0, t_c=np.inf, dt=DT, P=FIXED):
    """Expected weekly reported cases for theta = (beta, rho, I0)."""
    beta, rho, I0 = theta
    ts, ys = simulate(beta, I0, n_weeks, p=p, t_c=t_c, dt=dt, y0=initial_state_growth(beta, I0, P), P=P)
    return weekly_cases(ts, ys, rho)

### 3.4 Observation model and Metropolis sampler

Reported cases in week $w$ are negative binomial with mean $\mu_w = \rho N \Delta c_w$ and dispersion $k = 20$, so $\text{Var}(y_w) = \mu_w + \mu_w^2/k$.

The sampler works in $\phi = (\log\beta,\ \text{logit}\,\rho,\ \log I_0)$, so proposals never leave the valid range. The priors are uniform on the natural scale, $\beta \in [0.05, 1.5]$, $\rho \in [0.001, 1]$, $I_0 \in [1, 10^5]$, so the log-Jacobian of the transform, $\log\beta + \log\rho + \log(1-\rho) + \log I_0$, is added to the log-posterior.

Procedure: a Nelder–Mead search finds the posterior mode; a 1,000–1,500-iteration pilot chain estimates the posterior covariance; the main chains then use Gaussian random-walk proposals with covariance $2.38^2/d$ times that estimate ($d = 3$). The proposal is fixed during the main runs, so they are ordinary Metropolis chains. Convergence is checked with the Gelman–Rubin $\hat R$ across two chains started from over-dispersed points, and with the effective sample size (ESS).

In [ ]:
K_NB = 20.0   # dispersion: Var = mu + mu^2 / k

def nb_logpmf(y, mu, k=K_NB):
    """log P(Y = y) for a negative binomial with mean mu and dispersion k."""
    mu = np.maximum(mu, 1e-9)
    return (gammaln(y + k) - gammaln(k) - gammaln(y + 1)
            + k * np.log(k / (k + mu)) + y * np.log(mu / (k + mu)))

In [ ]:
BOUNDS = {"beta": (0.05, 1.5), "rho": (0.001, 1.0), "I0": (1.0, 1e5)}   # uniform priors on the natural scale

def to_theta(phi):
    """phi = (log beta, logit rho, log I0) -> theta = (beta, rho, I0)."""
    return np.array([np.exp(phi[0]), 1 / (1 + np.exp(-phi[1])), np.exp(phi[2])])

def to_phi(theta):
    b, r, i = theta
    return np.array([np.log(b), np.log(r / (1 - r)), np.log(i)])

def log_post(phi, y=None, mask=None):
    """Log posterior in phi-space. The priors are uniform on (beta, rho, I0), so the Jacobian of the
    transform, log beta + log rho + log(1 - rho) + log I0, is added."""
    y = y_obs if y is None else y
    mask = obs_mask if mask is None else mask
    b, r, i = theta = to_theta(phi)
    for val, (lo, hi) in zip(theta, BOUNDS.values()):
        if not lo <= val <= hi:
            return -np.inf
    mu = expected_cases(theta)
    log_jac = np.log(b) + np.log(r) + np.log(1 - r) + np.log(i)
    return nb_logpmf(y[mask], mu[mask]).sum() + log_jac

def metropolis(logp, phi0, cov, n_iter, rng):
    """Random-walk Metropolis with Gaussian proposals N(phi, cov). Returns the chain and acceptance rate."""
    L = np.linalg.cholesky(cov)
    chain = np.empty((n_iter, len(phi0)))
    phi, lp, acc = np.array(phi0, float), logp(phi0), 0
    for t in range(n_iter):
        prop = phi + L @ rng.standard_normal(len(phi))
        lp_prop = logp(prop)
        if np.log(rng.uniform()) < lp_prop - lp:
            phi, lp, acc = prop, lp_prop, acc + 1
        chain[t] = phi
    return chain, acc / n_iter

def gelman_rubin(chains):
    """Potential scale reduction factor R-hat for each parameter; chains has shape (m, n, d)."""
    m, n, _ = chains.shape
    W = chains.var(axis=1, ddof=1).mean(axis=0)
    B = n * chains.mean(axis=1).var(axis=0, ddof=1)
    return np.sqrt(((n - 1) / n * W + B / n) / W)

def ess(x):
    """Effective sample size from the initial positive sequence of autocorrelations."""
    x = x - x.mean(); n = len(x)
    ac = np.correlate(x, x, "full")[n - 1:] / (x.var() * np.arange(n, 0, -1))
    s = 0.0
    for k in range(1, n - 1, 2):
        pair = ac[k] + ac[k + 1]
        if pair < 0:
            break
        s += pair
    return n / (1 + 2 * s)

def calibrate(logp, phi_start, n_pilot, n_main, burn, rng, n_chains=2, label=""):
    """MAP start -> pilot run to learn the proposal covariance -> n_chains main runs."""
    t0 = time.perf_counter()
    res = optimize.minimize(lambda p: -logp(p), phi_start, method="Nelder-Mead",
                            options={"xatol": 1e-4, "fatol": 1e-4, "maxiter": 2000})
    pilot, acc_p = metropolis(logp, res.x, np.diag([0.02, 0.05, 0.05])**2, n_pilot, rng)
    cov = np.cov(pilot[n_pilot // 2:].T) * 2.38**2 / 3 + 1e-10 * np.eye(3)
    chains, accs = [], []
    for c in range(n_chains):
        start = res.x + rng.multivariate_normal(np.zeros(3), 4 * cov)   # over-dispersed starts
        ch, a = metropolis(logp, start, cov, n_main, rng)
        chains.append(ch[burn:]); accs.append(a)
    chains = np.array(chains)
    rhat = gelman_rubin(chains)
    ess_ = [ess(chains[:, :, j].ravel()) for j in range(3)]
    print(f"{label}: pilot acceptance {acc_p:.2f}; main acceptance {', '.join(f'{a:.2f}' for a in accs)}; "
          f"R-hat {np.round(rhat, 3)}; ESS {np.round(ess_).astype(int)}; {time.perf_counter() - t0:.0f} s")
    return chains, accs, rhat

def summarise(samples, names=("beta", "rho", "I0")):
    q = np.percentile(samples, [2.5, 50, 97.5], axis=0)
    return pd.DataFrame(q.T, index=names, columns=["2.5%", "median", "97.5%"])

## 4. Verification

Each check ends in an `assert`, so a failure stops *Run all*.

**V1. Conservation.** Without a campaign, $s+e+i+r = 1$ and $V_s+V_e+V_i = m_0$ for all time. With a campaign from $t_c$, the mosquito total obeys $\dot V = m_0\mu_v(1-p) - \mu_v V$, so $V(t) = m_0\left[1 - p\left(1 - e^{-\mu_v (t - t_c)}\right)\right]$.

In [ ]:
ts, ys = simulate(BETA_TEST, I0_TEST, n_weeks=25)
err_h = np.max(np.abs(ys[:, :4].sum(axis=1) - 1))
err_v = np.max(np.abs(ys[:, 5:].sum(axis=1) - FIXED["m0"]))
print(f"max |s+e+i+r - 1| = {err_h:.2e},  max |Vs+Ve+Vi - m0| = {err_v:.2e}")
assert err_h < 1e-10 and err_v < 1e-10
print("PASS: humans and mosquitoes conserved (no campaign)")

# with a campaign the mosquito total relaxes to m0 (1 - p)
ts, ys = simulate(BETA_TEST, I0_TEST, n_weeks=40, p=0.4, t_c=35.0)
Vtot = ys[:, 5:].sum(axis=1)
expected = FIXED["m0"] * (1 - 0.4 * (1 - np.exp(-FIXED["mu_v"] * np.clip(ts - 35.0, 0, None))))
err = np.max(np.abs(Vtot - expected))
print(f"max |V_total - analytic m0(1 - p(1 - e^(-mu_v (t - t_c))))| = {err:.2e}")
assert err < 1e-8
print("PASS: mosquito total follows the analytic relaxation after the campaign starts")

**V2. Limiting case.** With $\beta = 0$ there is no transmission: cumulative incidence and infected mosquitoes stay at zero, and infectious humans decay as $i_0 e^{-\gamma t}$. The tolerance allows for RK4's own error on this exponential (about $10^{-7}$ relative at $\Delta t = 0.5$).

In [ ]:
ts, ys = simulate(0.0, I0_TEST, n_weeks=25)
assert np.max(ys[:, 4]) < 1e-15 and np.max(ys[:, 6:]) < 1e-15
i_exact = (I0_TEST / N) * np.exp(-FIXED["gamma"] * ts)
rel = np.max(np.abs(ys[:, 2] - i_exact)) / (I0_TEST / N)
print(f"max relative error vs i0 exp(-gamma t) = {rel:.1e}")
assert rel < 1e-5
print("PASS: beta = 0 gives no new infections, and i(t) decays as i0 exp(-gamma t)")

**V3. Independent $R_0$ check.** $R_0$ is recomputed numerically as the spectral radius of the next-generation matrix $K = F V^{-1}$ for the infected compartments $(e, i, V_e, V_i)$, and compared with the closed form.

In [ ]:
def R0_ngm(beta, P=FIXED):
    sh, g, sv, mv, m = P["sigma_h"], P["gamma"], P["sigma_v"], P["mu_v"], P["m0"]
    # infected compartments x = (e, i, Ve, Vi) at the disease-free state (s = 1, Vs = m0)
    F = np.array([[0, 0,      0, beta],
                  [0, 0,      0, 0   ],
                  [0, beta*m, 0, 0   ],
                  [0, 0,      0, 0   ]], dtype=float)
    V = np.array([[ sh, 0, 0,       0 ],
                  [-sh, g, 0,       0 ],
                  [ 0,  0, sv + mv, 0 ],
                  [ 0,  0, -sv,     mv]], dtype=float)
    K = F @ np.linalg.inv(V)
    return np.max(np.abs(np.linalg.eigvals(K)))

for b in [0.1, 0.2, 0.35, 0.5]:
    a, n_ = R0(b), R0_ngm(b)
    print(f"beta = {b:.2f}: closed form R0 = {a:.6f}, next-generation matrix = {n_:.6f}")
    assert abs(a - n_) < 1e-8
print("PASS: closed-form R0 equals the spectral radius of the next-generation matrix")

**V4. Analytic threshold.** Mosquitoes start at the reduced equilibrium $m_0(1-p)$ and the campaign runs from $t = 0$. Just below $p_c$ the late-time growth rate of infectious humans must be positive; just above $p_c$ it must be negative.

In [ ]:
def growth_rate(beta, p, days=600):
    """Late-time exponential growth rate of infectious humans, starting at the reduced mosquito equilibrium."""
    m = FIXED["m0"] * (1 - p)
    y0 = initial_state(1e-3, m=m)            # tiny seed keeps s close to 1 (linear regime)
    ts, ys = simulate(beta, 1e-3, n_weeks=days // 7, p=p, t_c=0.0, y0=y0)
    late = ts > ts[-1] - 100
    return np.polyfit(ts[late], np.log(ys[late, 2]), 1)[0]

pc = p_crit(BETA_TEST)
r_below, r_above = growth_rate(BETA_TEST, pc - 0.03), growth_rate(BETA_TEST, pc + 0.03)
print(f"p_c = {pc:.4f}: growth rate at p_c - 0.03 = {r_below:+.4f}/day, at p_c + 0.03 = {r_above:+.4f}/day")
assert r_below > 0 > r_above
print("PASS: the outbreak grows just below p_c and dies out just above it")

**V5. Time-step convergence.** Cumulative incidence at week 25 is compared with a reference RK4 solution at $\Delta t = 1/128$ day. The fitted log–log slopes must match the theoretical orders: 1 for Euler and 4 for RK4.

In [ ]:
def c_final(solver, dt):
    ts, ys = simulate(BETA_TEST, I0_TEST, n_weeks=25, dt=dt, solver=solver)
    return ys[-1, 4]

ref = c_final(rk4, 1/128)
dts = np.array([1.0, 0.5, 0.25, 0.125, 0.0625])
err_e = np.array([abs(c_final(euler, d) - ref) for d in dts])
err_r = np.array([abs(c_final(rk4, d) - ref) for d in dts])
slope_e = np.polyfit(np.log(dts), np.log(err_e), 1)[0]
slope_r = np.polyfit(np.log(dts), np.log(err_r), 1)[0]
print("dt      Euler error   RK4 error")
for d, a, b in zip(dts, err_e, err_r): print(f"{d:<7} {a:.3e}     {b:.3e}")
print(f"fitted order: Euler {slope_e:.2f} (expected 1), RK4 {slope_r:.2f} (expected 4)")
assert 0.8 < slope_e < 1.2 and 3.5 < slope_r < 4.5
print("PASS: measured orders match theory")

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(dts, err_e, "o-", label=f"Euler (slope {slope_e:.2f})")
ax.loglog(dts, err_r, "s-", label=f"RK4 (slope {slope_r:.2f})")
ax.loglog(dts, err_e[0] * dts / dts[0], "k--", lw=0.8, label="slope 1")
ax.loglog(dts, err_r[0] * (dts / dts[0])**4, "k:", lw=0.8, label="slope 4")
ax.set_xlabel("time step dt [day]"); ax.set_ylabel("|error| in cumulative incidence at week 25")
ax.set_title("Time-step convergence"); ax.legend(); fig.tight_layout()
fig.savefig("figures/convergence.png", dpi=150)
plt.show()

**V6. Calibration building blocks.** The eigenvector starting state must still conserve humans and mosquitoes, and the hand-written negative-binomial log-probability must match `scipy.stats.nbinom`. That check catches the common mistake of mixing up scipy's $(n, p)$ parameterisation with mean and dispersion.

In [ ]:
ts, ys = simulate(0.3, 2000, N_WEEKS, y0=initial_state_growth(0.3, 2000))
assert np.max(np.abs(ys[:, :4].sum(axis=1) - 1)) < 1e-10 and np.max(np.abs(ys[:, 5:].sum(axis=1) - FIXED["m0"])) < 1e-10
print("PASS: the eigenvector start still conserves humans and mosquitoes")

yy, mm = np.array([0, 3, 250, 1600]), np.array([5.0, 3.5, 300.0, 1400.0])
assert np.allclose(nb_logpmf(yy, mm), stats.nbinom.logpmf(yy, n=K_NB, p=K_NB / (K_NB + mm)), atol=1e-10)
print("PASS: own negative-binomial log-pmf matches scipy.stats.nbinom")

**V7. Parameter recovery.** Synthetic weekly cases are generated from known parameters $\theta_{\text{true}} = (0.22,\ 0.02,\ 5000)$, with the same weeks, the same missing weeks and negative-binomial noise. The sampler must recover all three true values inside their 95% credible intervals, with $\hat R < 1.1$. While building the notebook, this check was repeated on five independent synthetic datasets and passed every time.

In [ ]:
rng_syn = np.random.default_rng(SEED + 1)
THETA_TRUE = np.array([0.22, 0.02, 5000.0])
mu_true = expected_cases(THETA_TRUE)
y_syn = rng_syn.negative_binomial(K_NB, K_NB / (K_NB + mu_true)).astype(float)
y_syn[~obs_mask] = np.nan                           # same missing weeks as the real data
logp_syn = lambda phi: log_post(phi, y=y_syn)

chains_syn, acc_syn, rhat_syn = calibrate(logp_syn, to_phi([0.4, 0.1, 1000.0]), 1000, 3000, 750, rng, label="synthetic")
post_syn = np.array([to_theta(p) for p in chains_syn.reshape(-1, 3)])
tab = summarise(post_syn); tab["true"] = THETA_TRUE; print(tab)
inside = (tab["2.5%"] <= tab["true"]) & (tab["true"] <= tab["97.5%"])
assert inside.all(), "a true value lies outside its 95% credible interval"
assert (rhat_syn < 1.1).all()
print("PASS: all three true parameters lie inside their 95% credible intervals, R-hat < 1.1")

## 5. Experiments and results

### 5.1 Calibration to the Colombo 2026 wave

In [ ]:
chains, accs, rhat = calibrate(log_post, to_phi([0.4, 0.1, 1000.0]), 1500, 6000, 1500, rng, label="Colombo")
assert (rhat < 1.1).all(), "chains have not converged"
assert all(0.15 < a < 0.5 for a in accs), "acceptance rate outside the usual range"
post = np.array([to_theta(p) for p in chains.reshape(-1, 3)])
R0_post = R0(post[:, 0]); pc_post = p_crit(post[:, 0])
tab = summarise(np.column_stack([post, R0_post, pc_post]), names=("beta", "rho", "I0", "R0", "p_c"))
print(tab.round(4))
THETA_MED = np.median(post, axis=0)

fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
for j, name in enumerate(["log beta", "logit rho", "log I0"]):
    for c in range(chains.shape[0]):
        axes[j].plot(chains[c, :, j], lw=0.4)
    axes[j].set_ylabel(name)
axes[-1].set_xlabel("iteration after burn-in"); fig.suptitle("Trace plots (two chains)"); fig.tight_layout()
fig.savefig("figures/traces.png", dpi=150)
plt.show()

Posterior predictive check: 400 posterior draws are simulated, giving the 95% band of the model mean, and negative-binomial noise is added to give the 95% predictive band. A well-calibrated model should contain roughly 95% of the observed weeks in the predictive band.

In [ ]:
idx = rng.choice(len(post), 400, replace=False)
mu_draws = np.array([expected_cases(post[k]) for k in idx])
y_pred = rng.negative_binomial(K_NB, K_NB / (K_NB + mu_draws))
mean_lo, mean_med, mean_hi = np.percentile(mu_draws, [2.5, 50, 97.5], axis=0)
pred_lo, pred_hi = np.percentile(y_pred, [2.5, 97.5], axis=0)
coverage = np.mean((y_obs[obs_mask] >= pred_lo[obs_mask]) & (y_obs[obs_mask] <= pred_hi[obs_mask]))
print(f"Observed weeks inside the 95% posterior predictive band: {coverage:.0%}")

fig, ax = plt.subplots(figsize=(8, 4))
wk = fit.epi_week
ax.fill_between(wk, pred_lo, pred_hi, color="tab:blue", alpha=0.15, label="95% predictive band")
ax.fill_between(wk, mean_lo, mean_hi, color="tab:blue", alpha=0.35, label="95% band of model mean")
ax.plot(wk, mean_med, color="tab:blue", label="posterior median")
ax.plot(wk[obs_mask], y_obs[obs_mask], "ko", ms=4, label="reported cases")
ax.plot(wk[~obs_mask], fit.cases_raw[~obs_mask], "x", color="0.5", label="missing week (not fitted)")
ax.set_xlabel("epidemiological week, 2026"); ax.set_ylabel("cases per week")
ax.set_title("Calibrated model vs Colombo data"); ax.legend(fontsize=8); fig.tight_layout()
fig.savefig("figures/fit.png", dpi=150)
plt.show()

### 5.2 Campaign scenarios

*Campaign intensity × start-week grid, heat-map and sensitivity analysis: to be added.*

## 6. Discussion and limitations

*To be added.*

**Web demo reference value:** *to be added.*

## 7. References

*To be added (IEEE style).*